# 🔬 Ví dụ giải thích SIÊU DỄ: Vòng lặp Agentic (Tool Use)

> File bổ sung — giải thích phần khó nhất & nâng cao nhất của khóa học một cách dễ hiểu nhất có thể.

Nếu bạn từng đọc bài **Tool Use** mà thấy rối ở những câu hỏi này:
- *Tại sao phải gửi lại `response.content`?*
- *`tool_use_id` để làm cái gì?*
- *Tại sao kết quả công cụ lại bỏ vào message của `user` (mình) chứ không phải `assistant`?*
- *Vòng lặp `while True` thực ra đang lặp cái gì?*

...thì file này dành cho bạn. Ta sẽ **"chụp X-quang"** — in ra **chính xác** cái gì được gửi đi ở từng bước.


## 1. Phép so sánh đời thực: Trợ lý qua điện thoại ☎️

Hãy tưởng tượng Claude là một **trợ lý cực thông minh nhưng KHÔNG có tay chân**, bạn nói chuyện với anh ấy **qua điện thoại**:

| Trong đời thực | Trong code |
|---|---|
| Bạn gọi điện hỏi trợ lý | Bạn gửi `messages` cho Claude |
| Trợ lý không tự tra cứu được → nhờ bạn: *"Tra giúp tôi X, mã việc **#A1**"* | Claude trả về `stop_reason="tool_use"` + khối `tool_use` (id=`#A1`) |
| Bạn cúp máy, tự đi tra cứu | Code của bạn chạy hàm công cụ |
| Bạn gọi lại: *"Kết quả cho việc **#A1** là..."* | Bạn gửi `tool_result` với `tool_use_id="#A1"` |
| Trợ lý dùng kết quả → trả lời bạn | Claude trả về câu trả lời cuối (`end_turn`) |

### 🧠 Điểm mấu chốt gây rối nhất:
**Trợ lý này bị "mất trí nhớ" sau MỖI cuộc gọi** (API là *stateless*). Nên mỗi lần gọi lại, bạn phải **đọc lại TOÀN BỘ cuộc trò chuyện từ đầu** — đó là lý do danh sách `messages` cứ **phình to dần**, và bạn phải tự lưu nó.

➡️ `messages` chính là **"bộ nhớ" của cuộc trò chuyện** mà BẠN giữ hộ, không phải Claude giữ.


## 2. Chuẩn bị: 1 công cụ đơn giản + hàm in "X-quang" 

In [ ]:
import anthropic, json
client = anthropic.Anthropic()
MODEL = "claude-opus-4-8"

# Một công cụ duy nhất cho dễ theo dõi: máy tính
tools = [{
    "name": "calculator",
    "description": "Tính một biểu thức toán học.",
    "input_schema": {
        "type": "object",
        "properties": {"expression": {"type": "string"}},
        "required": ["expression"],
    },
}]

def run_calculator(expression):
    return str(eval(expression))   # chỉ demo, thực tế cần kiểm tra an toàn

In [ ]:
def x_quang(messages):
    """In danh sách messages một cách dễ đọc để THẤY nó phình to thế nào."""
    print("┌" + "─" * 60)
    for i, m in enumerate(messages):
        role = m["role"].upper()
        content = m["content"]
        if isinstance(content, str):
            print(f"│ [{i}] {role}: {content}")
        else:
            # content là danh sách các khối (block)
            print(f"│ [{i}] {role}:")
            for block in content:
                # block có thể là dict (do mình tạo) hoặc object (do Claude trả về)
                btype = block["type"] if isinstance(block, dict) else block.type
                if btype == "text":
                    text = block["text"] if isinstance(block, dict) else block.text
                    print(f"│      • text: {text[:70]}")
                elif btype == "tool_use":
                    name = block["name"] if isinstance(block, dict) else block.name
                    bid  = block["id"] if isinstance(block, dict) else block.id
                    inp  = block["input"] if isinstance(block, dict) else block.input
                    print(f"│      • tool_use (id={bid[-6:]}): gọi {name}({inp})")
                elif btype == "tool_result":
                    print(f"│      • tool_result (cho id={block['tool_use_id'][-6:]}): {block['content']}")
    print("└" + "─" * 60)

## 3. Chạy CHẬM từng bước — xem `messages` phình to

Thay vì để vòng lặp chạy vù một cái, ta sẽ chạy **từng bước một** và in `messages` sau mỗi bước. Hãy chú ý danh sách dài thêm thế nào.


### 🔹 BƯỚC 0: Bạn đặt câu hỏi

`messages` ban đầu chỉ có 1 phần tử: câu hỏi của bạn.


In [ ]:
messages = [
    {"role": "user", "content": "Tính giúp 1234 nhân 5678 rồi cộng 90."}
]

print("messages SAU bước 0:")
x_quang(messages)

### 🔹 BƯỚC 1: Gọi Claude → nó XIN gọi công cụ

Claude **chưa trả lời ngay**. Nó nhận ra cần tính toán → trả về `stop_reason="tool_use"` kèm "đơn đặt hàng" (khối `tool_use`) có một **mã id** riêng.


In [ ]:
response = client.messages.create(model=MODEL, max_tokens=1024, tools=tools, messages=messages)

print("stop_reason:", response.stop_reason, "  <-- 'tool_use' = Claude muốn gọi công cụ\n")
print("Claude trả về các khối:")
for b in response.content:
    if b.type == "tool_use":
        print(f"  → tool_use: gọi {b.name}({b.input}), id = {b.id}")
    elif b.type == "text":
        print(f"  → text: {b.text}")

### 🔹 BƯỚC 2: Lưu "đơn đặt hàng" của Claude vào lịch sử

❓ **Tại sao phải `append(response.content)`?**
Vì lần gọi sau Claude bị "mất trí nhớ", ta phải nhắc lại cho nó: *"Anh vừa nói anh muốn gọi calculator với id #A1 đấy."* Nếu không lưu, Claude sẽ không biết kết quả mình sắp đưa là để trả lời cho cái gì.

➡️ Lưu **nguyên vẹn** `response.content` (chứa khối `tool_use`) vào lịch sử, vai trò `assistant`.


In [ ]:
messages.append({"role": "assistant", "content": response.content})

print("messages SAU bước 2 (đã dài thêm 1 — lời 'xin gọi công cụ' của Claude):")
x_quang(messages)

### 🔹 BƯỚC 3: Bạn thực thi công cụ & gửi kết quả về

❓ **Tại sao `tool_result` lại nằm trong message `user`?**
Vì theo luật xen kẽ user→assistant→user..., sau lượt `assistant` (Claude xin gọi) phải tới lượt **`user`** (bạn). Bạn — người vận hành — là bên "giao kết quả" về, nên kết quả đi trong message `user`.

❓ **`tool_use_id` để làm gì?**
Như **mã đơn hàng** 🎫. Khi Claude gọi 3 công cụ cùng lúc, mỗi kết quả phải dán đúng mã để Claude biết "kết quả này trả lời cho yêu cầu nào". Phải **khớp đúng** `id` ở khối `tool_use`.


In [ ]:
# Lấy khối tool_use ra, chạy hàm thật, gói kết quả lại
tool_results = []
for b in response.content:
    if b.type == "tool_use":
        ket_qua = run_calculator(b.input["expression"])
        print(f"  🔧 Mình tự tính: {b.input['expression']} = {ket_qua}")
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": b.id,        # <-- DÁN ĐÚNG MÃ
            "content": ket_qua,
        })

messages.append({"role": "user", "content": tool_results})

print("\nmessages SAU bước 3 (dài thêm — kết quả công cụ mình gửi về):")
x_quang(messages)

### 🔹 BƯỚC 4: Gọi Claude LẦN NỮA → giờ nó trả lời

Ta gửi **lại toàn bộ** `messages` (giờ đã có cả: câu hỏi + lời xin gọi công cụ + kết quả). Claude đọc hết, thấy đã có kết quả → trả lời cuối cùng với `stop_reason="end_turn"`.


In [ ]:
final = client.messages.create(model=MODEL, max_tokens=1024, tools=tools, messages=messages)

print("stop_reason:", final.stop_reason, "  <-- 'end_turn' = đã trả lời xong, dừng vòng lặp\n")
print("💬 Trả lời cuối:", final.content[0].text)

## 4. Ghép lại = Vòng lặp `while True`

Giờ nhìn lại 4 bước trên, bạn sẽ thấy chúng lặp đi lặp lại: **gọi Claude → nếu nó xin công cụ thì chạy & gửi kết quả → gọi lại**. Đó chính là `while True`. Vòng lặp chỉ dừng khi `stop_reason == "end_turn"`.

```
while True:
    response = gọi_claude(messages)

    if response.stop_reason == "end_turn":
        return câu_trả_lời        # ✅ XONG, thoát

    messages.append(lời_xin_công_cụ_của_claude)   # bước 2
    messages.append(kết_quả_công_cụ_của_bạn)      # bước 3
    # ...rồi quay lại đầu while: gọi claude lần nữa (bước 4)
```

Vòng lặp tự lo được cả trường hợp Claude cần gọi công cụ **nhiều lần liên tiếp** (tính A xong, dựa vào A gọi tiếp B...).


In [ ]:
def agentic_loop(cau_hoi):
    messages = [{"role": "user", "content": cau_hoi}]
    vong = 0
    while True:
        vong += 1
        response = client.messages.create(model=MODEL, max_tokens=1024,
                                          tools=tools, messages=messages)
        print(f"  (vòng {vong}: stop_reason = {response.stop_reason})")

        if response.stop_reason == "end_turn":
            return next(b.text for b in response.content if b.type == "text")

        messages.append({"role": "assistant", "content": response.content})
        results = []
        for b in response.content:
            if b.type == "tool_use":
                results.append({"type": "tool_result", "tool_use_id": b.id,
                                "content": run_calculator(b.input["expression"])})
        messages.append({"role": "user", "content": results})

print("Câu hỏi: (15 + 5) nhân 3, rồi lấy kết quả đó chia 4 là bao nhiêu?\n")
print("💬", agentic_loop("Tính (15 + 5) nhân 3, rồi lấy kết quả đó chia cho 4."))

## 5. 3 lỗi thường gặp khiến vòng lặp "vỡ" 💥

| Lỗi | Hậu quả | Cách đúng |
|---|---|---|
| **Quên** `append(response.content)` | Claude không biết kết quả trả lời cho gì → lỗi/lú | Luôn lưu nguyên `response.content` |
| `tool_use_id` **không khớp** | API báo lỗi (kết quả mồ côi) | Copy đúng `id` từ khối `tool_use` |
| Bỏ `tool_result` vào message **`assistant`** | Sai luật xen kẽ → lỗi | Kết quả công cụ luôn nằm trong message **`user`** |

> 💡 Mẹo nhớ: **`user` luôn là bên "đưa thông tin vào"** (câu hỏi HOẶC kết quả công cụ); **`assistant` luôn là Claude** (trả lời HOẶC xin gọi công cụ).


## 6. Vì sao hiểu cái này lại quan trọng? 🌟

Vì **TẤT CẢ** những thứ "xịn" sau đều chỉ là vòng lặp này + công cụ khác nhau:

| Bài học | Thực chất là vòng lặp agentic với... |
|---|---|
| **Agentic Search** (bài 5) | ...công cụ `search_documents` |
| **Agents** (bài 8) | ...nhiều công cụ phối hợp |
| **MCP** (bài 6) | ...công cụ đến từ MCP server bên ngoài |
| **Claude Code** (bài 7) | ...công cụ `bash`, `read`, `edit`... |
| **Computer Use** (bài 7) | ...công cụ "click chuột / gõ phím" |

👉 Nắm chắc vòng lặp này = bạn hiểu được **mọi agent** trên đời, dù phức tạp đến đâu.


---
## ✅ Tóm tắt 1 dòng

> **Agentic loop** = lặp lại {gọi Claude → nếu nó xin công cụ thì bạn chạy & gửi kết quả về} cho đến khi nó nói "xong" (`end_turn`). `messages` là bộ nhớ bạn giữ hộ; `tool_use_id` là mã đơn hàng để ghép đúng kết quả.
